<a href="https://colab.research.google.com/github/solitude-ui/tensorflow-vs-pytorch-deep-learning/blob/main/Assesment_Pytorch_vs_Tensorflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Pytorch vs Tensorflow**



> Income value predictor



#Load Libraries

In [108]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.impute import SimpleImputer,KNNImputer
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score,classification_report, confusion_matrix

#Load Datasets

In [109]:
df=pd.read_csv("/content/drive/MyDrive/ICTK - AI-ML/Datasets/adult_dataset.csv")

#EDA analysis

In [110]:
df.head(5)

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39.0,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40.0,NaN,<=50K
1,50.0,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13.0,United-States,<=50K
2,38.0,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40.0,United-States,<=50K
3,53.0,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40.0,United-States,<=50K
4,28.0,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40.0,Cuba,<=50K


In [111]:
df.shape

(48842, 15)

In [112]:
df.describe()

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,46400.000000,4.884200e+04,48842.000000,48842.000000,48842.000000,46400.000000
mean,38.679116,1.896641e+05,10.078089,1079.067626,87.502314,40.448599
std,13.952954,1.056040e+05,2.570973,7452.019058,403.004552,12.646992
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.175505e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.781445e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.376420e+05,12.000000,0.000000,0.000000,45.000000
max,255.000000,1.490400e+06,16.000000,99999.000000,4356.000000,250.000000


In [113]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             46400 non-null  float64
 1   workclass       45480 non-null  object 
 2   fnlwgt          48842 non-null  int64  
 3   education       48842 non-null  object 
 4   education-num   48842 non-null  int64  
 5   marital-status  48842 non-null  object 
 6   occupation      45485 non-null  object 
 7   relationship    48842 non-null  object 
 8   race            48842 non-null  object 
 9   sex             48842 non-null  object 
 10  capital-gain    48842 non-null  int64  
 11  capital-loss    48842 non-null  int64  
 12  hours-per-week  46400 non-null  float64
 13  native-country  46137 non-null  object 
 14  income          48842 non-null  object 
dtypes: float64(2), int64(4), object(9)
memory usage: 5.6+ MB


#Data Preprocessing

## Handle Duplicates

In [114]:
df.drop_duplicates()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39.0,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40.0,NaN,<=50K
1,50.0,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13.0,United-States,<=50K
2,38.0,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40.0,United-States,<=50K
3,53.0,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40.0,United-States,<=50K
4,28.0,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40.0,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,39.0,Private,215419,Bachelors,13,Divorced,Prof-specialty,Not-in-family,White,Female,0,0,36.0,United-States,<=50K.
48838,64.0,NaN,321403,HS-grad,9,Widowed,NaN,Other-relative,Black,Male,0,0,40.0,United-States,<=50K.
48839,38.0,Private,374983,Bachelors,13,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,50.0,United-States,<=50K.
48840,44.0,Private,83891,Bachelors,13,Divorced,Adm-clerical,Own-child,Asian-Pac-Islander,Male,5455,0,40.0,United-States,<=50K.


In [115]:
df.duplicated().sum()

np.int64(16)

## Handle null values

In [116]:
(df.isna().sum())/len(df)*100

,0
age,4.999795
workclass,6.883420
fnlwgt,0.000000
education,0.000000
education-num,0.000000
marital-status,0.000000
occupation,6.873183
relationship,0.000000
race,0.000000
sex,0.000000


**split the data into categorical and numerical**

In [117]:
categorical=df.select_dtypes(include="object")

In [118]:
numerical=df.select_dtypes(exclude="object")

### Multivariate-KNNImputer

In [119]:
knn_imputer=KNNImputer(n_neighbors=4)

In [120]:
numerical_imputed=pd.DataFrame(knn_imputer.fit_transform(numerical),columns=numerical.columns,index=numerical.index)

###Univariate- Simple imputer

In [121]:
simple_impute=SimpleImputer(strategy="most_frequent")

In [122]:
categorical_imputed=pd.DataFrame(simple_impute.fit_transform(categorical),columns=categorical.columns,index=categorical.index)

## Encode the data


In [123]:
cat_label=["race","sex","income"]

In [124]:
cat_onehot=['workclass', 'education', 'marital-status', 'occupation',
       'relationship', 'native-country']

### Label Encoder

In [125]:
lb=LabelEncoder()

In [126]:
for cols in cat_label:
  categorical_imputed[cols]=lb.fit_transform(categorical_imputed[cols])

### One-hot Encoding

In [127]:
oh=OneHotEncoder(sparse_output=False)

In [128]:
for cols in cat_onehot:
  encoded_values=oh.fit_transform(df[[cols]])
  encoded_cols=oh.get_feature_names_out()
  categorical_imputed[encoded_cols]=encoded_values
  categorical_imputed.drop(columns=cols,inplace=True)

/tmp/ipykernel_3533/3805131939.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  categorical_imputed[encoded_cols]=encoded_values
/tmp/ipykernel_3533/3805131939.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  categorical_imputed[encoded_cols]=encoded_values


## Concat category and numerical


In [129]:
df_clean=pd.concat([categorical_imputed,numerical_imputed],axis=1)

#Data Splitting

In [130]:
X=df_clean.drop(columns="income")
y=df_clean["income"]

In [131]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.2,stratify=y)

#Feature Scaling

In [132]:
scaler=StandardScaler()

In [133]:
X_train_scale=scaler.fit_transform(X_train)

In [134]:
X_test_scale=scaler.transform(X_test)

#class imbalance

In [135]:
y.value_counts()

,count
income,
0,24720
1,12435
2,7841
3,3846


In [136]:
smote=SMOTE(random_state=42)

In [137]:
X_train,y_train=smote.fit_resample(X_train_scale,y_train)

#Implement **Pytorch**

## Load Libraries

In [138]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader
from sklearn.metrics import accuracy_score,classification_report


## create tensor Dataset

In [139]:
y_test.dtypes

dtype('int64')

In [140]:
#create tensor Dataset
train_datasets =TensorDataset(
    torch.tensor(X_train,dtype=torch.float32),
    torch.tensor(y_train,dtype=torch.int64)
)

test_datasets =TensorDataset(
    torch.tensor(X_test.to_numpy(),dtype=torch.float32),
    torch.tensor(y_test.to_numpy(),dtype=torch.int64)
)

## create a Dataloader

In [141]:
#create a Dataloader

batch_size=32
train_loader=DataLoader(
    train_datasets,
    batch_size=batch_size,
    shuffle=True
)

test_loader=DataLoader(
    test_datasets,
    batch_size=batch_size,
    shuffle=True
)

## **Create Neural Network**

In [142]:
#importing tensor-nn when our class inherit the nn the normal python class become neural network

class IncomeFindModel(nn.Module):

  def __init__(self,input_features,hidden1_size,hidden2_size,num_classes,dropout_rate):
    super().__init__()

    self.network=nn.Sequential(
        #first layer
        nn.Linear(input_features,hidden1_size),
        nn.ReLU(),
        nn.Dropout(dropout_rate),

        #second layer
        nn.Linear(hidden1_size,hidden2_size),
        nn.ReLU(),
        nn.Dropout(dropout_rate),

        #Third Layer
        nn.Linear(hidden2_size,num_classes)
    )

  #forward propagation (aligned properly with __init__)
  def forward(self,x):
    return self.network(x)

## Get input feature and number of class

In [143]:
#Get input feature and number of class

input_features=X_train.shape[1]
num_classes=y_train.nunique()

print("Input features: ",input_features)
print("Number of classes: ",num_classes)

Input features:  106
Number of classes:  4


##Create model

In [144]:
#Create model
model=IncomeFindModel(
    input_features,
    num_classes=num_classes,
    hidden1_size=64,
    hidden2_size=32,
    dropout_rate=0.3
)

##Loss Function

In [145]:
loss_function=nn.CrossEntropyLoss()

## optimizer

In [146]:
optimizer=torch.optim.Adam(
    model.parameters(),
    lr=0.001 #learning rate
)

## Train

In [147]:
# Entire Training Loop
epochs = 100

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        # Forward pass
        predictions = model(X_batch)

        # Calculate Loss
        loss = loss_function(predictions, y_batch)

        # Store loss immediately
        total_loss += loss.item()

        # Remove previous gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

    average_loss = total_loss / len(train_loader)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {average_loss:.4f}")

Epoch [10/100], Loss: 0.9778
Epoch [20/100], Loss: 0.9613
Epoch [30/100], Loss: 0.9530
Epoch [40/100], Loss: 0.9436
Epoch [50/100], Loss: 0.9398
Epoch [60/100], Loss: 0.9381
Epoch [70/100], Loss: 0.9347
Epoch [80/100], Loss: 0.9333
Epoch [90/100], Loss: 0.9311
Epoch [100/100], Loss: 0.9278


## Testing

In [148]:
model.eval()  # To evaluate the model

all_actual = []  # To store actual values
all_predictions = []  # To store predicted values

with torch.no_grad():  # Disable gradient calculation for testing
    for X_batch, y_batch in test_loader:
        # Get predictions using parentheses, not square brackets
        predictions = model(X_batch)

        # Get predicted class
        predicted_classes = torch.argmax(predictions, dim=1)

        # Store actual and predicted classes
        all_actual.extend(y_batch.numpy())
        all_predictions.extend(predicted_classes.numpy())

## Evaluation

In [149]:


#Calculate accuracy
accuracy=accuracy_score(all_actual,all_predictions)
print("Accuracy :",accuracy)

#Classification report
print("\nClassification Report")
print(classification_report(all_actual,all_predictions))

Accuracy : 0.5133585832736206

Classification Report
              precision    recall  f1-score   support

           0       0.51      1.00      0.68      4945
           1       0.00      0.00      0.00      2487
           2       0.64      0.05      0.09      1568
           3       0.00      0.00      0.00       769

    accuracy                           0.51      9769
   macro avg       0.29      0.26      0.19      9769
weighted avg       0.36      0.51      0.36      9769



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Implement **Tensorflow**

##Load Libraries

In [150]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

##create Neural Network

In TensorFlow, Keras handles the dataset batching and training loop, so there is no need to create a TensorDataset and DataLoader.

In [151]:
# Get input features and number of classes

input_features = X_train.shape[1]
num_classes = len(set(y_train))

print("Input features:", input_features)
print("Number of classes:", num_classes)

Input features: 106
Number of classes: 4


##Create the model

In [152]:
# Create neural network model

model_tf = keras.Sequential([
    # Input layer
    keras.Input(shape=(input_features,)),

    # First layer
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),

    # Second layer
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),

    # Output layer
    layers.Dense(num_classes, activation="softmax")
])

model_tf.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │         6,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,060 (35.39 KB)

 Trainable params: 9,060 (35.39 KB)

 Non-trainable params: 0 (0.00 B)

##Loss Function

In [153]:
loss_function = "sparse_categorical_crossentropy"

##Optimiser

In [154]:
optimizer = keras.optimizers.Adam(
    learning_rate=0.001
)

model_tf.compile(
    optimizer=optimizer,
    loss=loss_function,
    metrics=["accuracy"]
)


##Train

In [155]:
# Entire training process

epochs = 100

history = model_tf.fit(
    X_train,
    y_train,
    epochs=epochs,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/100
1978/1978 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.5139 - loss: 1.0416 - val_accuracy: 0.0000e+00 - val_loss: 2.1371
Epoch 2/100
1978/1978 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5406 - loss: 0.9532 - val_accuracy: 0.0000e+00 - val_loss: 2.1318
Epoch 3/100
1978/1978 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5446 - loss: 0.9335 - val_accuracy: 1.8963e-04 - val_loss: 2.1831
Epoch 4/100
1978/1978 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.5514 - loss: 0.9231 - val_accuracy: 2.5284e-04 - val_loss: 2.2252
Epoch 5/100
1978/1978 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5523 - loss: 0.9189 - val_accuracy: 2.5284e-04 - val_loss: 2.2045
Epoch 6/100
1978/1978 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5540 - loss: 0.9143 - val_accuracy: 7.5853e-04 - val_loss: 2.2196
Epoch 7/100
1978/1978 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.5543 - loss: 0.9117 - val_accuracy: 4.4248e-04 - val_loss: 2.2002
Epoch 8/100
1978/1978 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step 

##Evaluation

In [156]:
# Evaluate the TensorFlow model on the held-out test set
y_pred_tf = np.argmax(model_tf.predict(X_test, verbose=0), axis=1)
y_test_np = np.asarray(y_test, dtype=np.int64)

print("TensorFlow Accuracy:", accuracy_score(y_test_np, y_pred_tf))
print("\nTensorFlow Classification Report:")
print(classification_report(y_test_np, y_pred_tf, zero_division=0))

TensorFlow Accuracy: 0.5283038182004299

TensorFlow Classification Report:
              precision    recall  f1-score   support

           0       0.53      0.98      0.69      4945
           1       0.00      0.00      0.00      2487
           2       0.52      0.19      0.28      1568
           3       0.00      0.00      0.00       769

    accuracy                           0.53      9769
   macro avg       0.26      0.29      0.24      9769
weighted avg       0.35      0.53      0.39      9769



# Evaluate both models

In [157]:
# Compare both baseline models on the same held-out test set.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_test_np = np.asarray(y_test, dtype=np.int64)
y_pred_torch = np.asarray(all_predictions, dtype=np.int64)

def metric_row(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (macro)": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1 (macro)": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "F1 (weighted)": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

baseline_results = pd.DataFrame([
    metric_row("TensorFlow (baseline)", y_test_np, y_pred_tf),
    metric_row("PyTorch (baseline)", y_test_np, y_pred_torch),
])
display(baseline_results.round(4))

print("\nPyTorch Classification Report:")
print(classification_report(y_test_np, y_pred_torch, zero_division=0))
print("\nTensorFlow Classification Report:")
print(classification_report(y_test_np, y_pred_tf, zero_division=0))

,Model,Accuracy,Precision (macro),Recall (macro),F1 (macro),F1 (weighted)
0,TensorFlow (baseline),0.5283,0.2625,0.2941,0.2427,0.3934
1,PyTorch (baseline),0.5025,0.1732,0.2506,0.1739,0.3432



PyTorch Classification Report:
              precision    recall  f1-score   support

           0       0.51      0.99      0.67      4945
           1       0.00      0.00      0.00      2487
           2       0.19      0.01      0.03      1568
           3       0.00      0.00      0.00       769

    accuracy                           0.50      9769
   macro avg       0.17      0.25      0.17      9769
weighted avg       0.29      0.50      0.34      9769


TensorFlow Classification Report:
              precision    recall  f1-score   support

           0       0.53      0.98      0.69      4945
           1       0.00      0.00      0.00      2487
           2       0.52      0.19      0.28      1568
           3       0.00      0.00      0.00       769

    accuracy                           0.53      9769
   macro avg       0.26      0.29      0.24      9769
weighted avg       0.35      0.53      0.39      9769



# perform Hyperparameter tuning-**OPTUNA**

Optuna is an open-source hyperparameter optimization framework in Python that automates tuning parameters (like learning rate, batch size, and network depth) for machine learning and deep learning models.

In [158]:
!pip install optuna
import optuna

In [160]:
#split data to training and validation sets
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

In [161]:
# 2 convert to pytorch tensors

train_data = TensorDataset(
    torch.tensor(X_train_split, dtype=torch.float32),
    torch.tensor(y_train_split, dtype=torch.int64)
)

val_data = TensorDataset(
    torch.tensor(X_test.values, dtype=torch.float32),
    torch.tensor(y_test.to_numpy(), dtype=torch.int64)
)

input_features = X_train.shape[1]
# Explicitly set num_classes to 3 (or length of unique classes in target)
num_classes = len(np.unique(y_train))

## Optuna for pytorch

In [162]:
#create objective function for optuna

def objective(trial):
  lr=trial.suggest_categorical("lr",[0.01,0.001,0.1]) #learning rate
  batch_size=trial.suggest_categorical("batch_size",[16,32,64,128])#BAtch Size
  hidden1=trial.suggest_categorical("hidden1",[32,64,128])#Neurons in layer 1
  hidden2=trial.suggest_categorical("hidden2",[16,32,64])#Neurons in layer 2
  dropout=trial.suggest_float("dropout",0.0,0.3,step=0.1) #Dropout rate

  #Loading the train and validation datasets with optimized batch size value
  train_loader=DataLoader(train_data,batch_size=batch_size,shuffle=False)
  val_loader=DataLoader(val_data,batch_size=batch_size,shuffle=False)

  #create Model with optimized hidden layer values,learning rate and droputs
  model=IncomeFindModel(input_features,hidden1,hidden2,num_classes,dropout)
  criterion=nn.CrossEntropyLoss()
  optimizer=torch.optim.Adam(model.parameters(),lr=lr)

  #Tuning the model
  epochs=20
  for epoch in range(epochs):
    model.train()
    for X_batch,y_batch in train_loader:
      optimizer.zero_grad()
      predictions=model(X_batch)
      loss=criterion(predictions,y_batch)
      loss.backward()#backward propogation
      optimizer.step()#updating weights

    #calculate Accuracy on validation Data
    model.eval()
    val_predictions=[]
    val_actual=[]

    with torch.no_grad():
      for X_batch,y_batch in val_loader:
        preds=model(X_batch)
        predicted_classes=torch.argmax(preds,dim=1)
        val_predictions.extend(predicted_classes.numpy())
        val_actual.extend(y_batch.numpy())

    val_accuracy=accuracy_score(val_actual,val_predictions)

  #return accuracy so optuna knows how good thesse parameters are
  return val_accuracy

In [163]:
#Passing the objective function inside the optune inorder find best parameter

study=optuna.create_study(direction="maximize")
study.optimize(objective,n_trials=20)

for key, value in study.best_params.items():
  print(f"{key}:{value}")
print(f"\nBest Validation Accuracy:{study.best_value:.4f}")

[I 2026-10-21 07:43:22,709] A new study created in memory with name: no-name-5df169e9-422b-45f2-bc23-0ddf2ee458b5
[I 2026-10-21 07:43:50,875] Trial 0 finished with value: 0.5125396662913297 and parameters: {'lr': 0.01, 'batch_size': 128, 'hidden1': 32, 'hidden2': 32, 'dropout': 0.1}. Best is trial 0 with value: 0.5125396662913297.
[I 2026-10-21 07:44:22,176] Trial 1 finished with value: 0.5112089261951069 and parameters: {'lr': 0.001, 'batch_size': 64, 'hidden1': 128, 'hidden2': 16, 'dropout': 0.0}. Best is trial 0 with value: 0.5125396662913297.
[I 2026-10-21 07:44:38,633] Trial 2 finished with value: 0.16050772852902037 and parameters: {'lr': 0.1, 'batch_size': 128, 'hidden1': 32, 'hidden2': 16, 'dropout': 0.0}. Best is trial 0 with value: 0.5125396662913297.
[I 2026-10-21 07:46:04,068] Trial 3 finished with value: 0.15825570682772033 and parameters: {'lr': 0.01, 'batch_size': 16, 'hidden1': 128, 'hidden2': 32, 'dropout': 0.1}. Best is trial 0 with value: 0.5125396662913297.
[I 2026-

lr:0.001
batch_size:32
hidden1:64
hidden2:16
dropout:0.2

Best Validation Accuracy:0.5187


In [164]:
# Fix the key in the study results from 'droput' to 'dropout' without rerunning the study
if "droput" in study.best_params:
    study.best_params["dropout"] = study.best_params.pop("droput")

In [166]:
# Train the final model with best parameters
best = study.best_params

# Ensure both spellings are covered in the local dictionary to prevent KeyError
if "droput" in best and "dropout" not in best:
    best["dropout"] = best["droput"]

# 1. Prepare full training and test dataloaders
full_train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.int64)
    ),
    batch_size=best["batch_size"],
    shuffle=True
)

test_loader = DataLoader(test_datasets, batch_size=best["batch_size"], shuffle=False)

# Build model with best parameters
final_model = IncomeFindModel(
    input_features=input_features,
    hidden1_size=best["hidden1"],
    hidden2_size=best["hidden2"],
    num_classes=num_classes,
    dropout_rate=best["dropout"]
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(final_model.parameters(), lr=best["lr"])

# 3. Train final model for 50 epochs
epochs = 50
print("\nTraining final model...")
for epoch in range(epochs):
    final_model.train()
    for X_batch, y_batch in full_train_loader:
        optimizer.zero_grad()
        loss = criterion(final_model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

# 4. Evaluate on Test Data
final_model.eval()
torch_test_preds = []
torch_test_actual = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = final_model(X_batch)
        predicted_classes = torch.argmax(preds, dim=1)
        torch_test_preds.extend(predicted_classes.numpy())
        torch_test_actual.extend(y_batch.numpy())

# Final Results

print("\nFINAL TEST EVALUATION \n")
print("Test Accuracy:", accuracy_score(torch_test_actual,torch_test_preds))
print("\nClassification Report:")
print(classification_report(torch_test_actual,torch_test_preds))


Training final model...

FINAL TEST EVALUATION 

Test Accuracy: 0.5112089261951069

Classification Report:
              precision    recall  f1-score   support

           0       0.51      1.00      0.68      4945
           1       0.00      0.00      0.00      2487
           2       0.64      0.03      0.06      1568
           3       0.12      0.01      0.01       769

    accuracy                           0.51      9769
   macro avg       0.32      0.26      0.19      9769
weighted avg       0.37      0.51      0.35      9769



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Optuna for TensorFlow

This section uses the same training/validation split and the same held-out test set as PyTorch. The test set is only used after tuning.

In [167]:
# Prepare NumPy arrays for TensorFlow.
X_train_np = np.asarray(X_train, dtype=np.float32)
X_test_np = np.asarray(X_test, dtype=np.float32)
y_train_np = np.asarray(y_train, dtype=np.int64)
y_test_np = np.asarray(y_test, dtype=np.int64)
X_train_split_np = np.asarray(X_train_split, dtype=np.float32)
X_val_split_np = np.asarray(X_val_split, dtype=np.float32)
y_train_split_np = np.asarray(y_train_split, dtype=np.int64)
y_val_split_np = np.asarray(y_val_split, dtype=np.int64)

def objective_tf(trial):
    tf.keras.backend.clear_session()
    tf.random.set_seed(42)

    lr = trial.suggest_categorical("lr", [0.01, 0.001, 0.0001])
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
    hidden1 = trial.suggest_categorical("hidden1", [32, 64, 128])
    hidden2 = trial.suggest_categorical("hidden2", [16, 32, 64])
    dropout = trial.suggest_float("dropout", 0.0, 0.3, step=0.1)

    model = keras.Sequential([
        keras.Input(shape=(input_features,)),
        layers.Dense(hidden1, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(hidden2, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    model.fit(
        X_train_split_np, y_train_split_np,
        validation_data=(X_val_split_np, y_val_split_np),
        epochs=20,
        batch_size=batch_size,
        verbose=0
    )

    _, val_accuracy = model.evaluate(
        X_val_split_np, y_val_split_np, verbose=0
    )
    return float(val_accuracy)

study_tf = optuna.create_study(direction="maximize")
study_tf.optimize(objective_tf, n_trials=20)

best_tf_params = study_tf.best_params.copy()
print("Best TensorFlow parameters:", best_tf_params)
print(f"Best TensorFlow validation accuracy: {study_tf.best_value:.4f}")


[I 2026-10-21 08:01:39,383] A new study created in memory with name: no-name-a8478f91-1ebb-4777-b89e-72a02f6117fc
[I 2026-10-21 08:02:05,567] Trial 0 finished with value: 0.46283185482025146 and parameters: {'lr': 0.01, 'batch_size': 128, 'hidden1': 64, 'hidden2': 64, 'dropout': 0.2}. Best is trial 0 with value: 0.46283185482025146.
[I 2026-10-21 08:03:18,134] Trial 1 finished with value: 0.49709227681159973 and parameters: {'lr': 0.0001, 'batch_size': 32, 'hidden1': 32, 'hidden2': 32, 'dropout': 0.0}. Best is trial 1 with value: 0.49709227681159973.
[I 2026-10-21 08:04:02,732] Trial 2 finished with value: 0.45385587215423584 and parameters: {'lr': 0.01, 'batch_size': 64, 'hidden1': 128, 'hidden2': 64, 'dropout': 0.2}. Best is trial 1 with value: 0.49709227681159973.
[I 2026-10-21 08:04:33,554] Trial 3 finished with value: 0.5850821733474731 and parameters: {'lr': 0.001, 'batch_size': 128, 'hidden1': 128, 'hidden2': 64, 'dropout': 0.1}. Best is trial 3 with value: 0.5850821733474731.
[

Best TensorFlow parameters: {'lr': 0.001, 'batch_size': 32, 'hidden1': 128, 'hidden2': 64, 'dropout': 0.0}
Best TensorFlow validation accuracy: 0.6205


In [168]:
# Refit the selected TensorFlow architecture on all training data.
tf.keras.backend.clear_session()
tf.random.set_seed(42)

model_tf_tuned = keras.Sequential([
    keras.Input(shape=(input_features,)),
    layers.Dense(best_tf_params["hidden1"], activation="relu"),
    layers.Dropout(best_tf_params["dropout"]),
    layers.Dense(best_tf_params["hidden2"], activation="relu"),
    layers.Dropout(best_tf_params["dropout"]),
    layers.Dense(num_classes, activation="softmax")
])

model_tf_tuned.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=best_tf_params["lr"]
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_tf_tuned.fit(
    X_train_np, y_train_np,
    epochs=50,
    batch_size=best_tf_params["batch_size"],
    verbose=0
)

y_pred_tf_tuned = np.argmax(
    model_tf_tuned.predict(X_test_np, verbose=0), axis=1
)

print("TensorFlow tuned test accuracy:",
      accuracy_score(y_test_np, y_pred_tf_tuned))
print("\\nTensorFlow tuned classification report:")
print(classification_report(y_test_np, y_pred_tf_tuned, zero_division=0))


TensorFlow tuned test accuracy: 0.5234926809294708
\nTensorFlow tuned classification report:
              precision    recall  f1-score   support

           0       0.52      1.00      0.68      4945
           1       0.00      0.00      0.00      2487
           2       0.60      0.12      0.20      1568
           3       0.00      0.00      0.00       769

    accuracy                           0.52      9769
   macro avg       0.28      0.28      0.22      9769
weighted avg       0.36      0.52      0.38      9769



#Evaluation after Hyperparameter tuning

In [169]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results_after_optuna = pd.DataFrame([
    {
        "Model": "PyTorch + Optuna",
        "Accuracy": accuracy_score(y_test_np, torch_test_preds),
        "Precision (macro)": precision_score(y_test_np, torch_test_preds, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test_np, torch_test_preds, average="macro", zero_division=0),
        "F1 (macro)": f1_score(y_test_np, torch_test_preds, average="macro", zero_division=0),
        "F1 (weighted)": f1_score(y_test_np, torch_test_preds, average="weighted", zero_division=0)
    },
    {
        "Model": "TensorFlow + Optuna",
        "Accuracy": accuracy_score(y_test_np, y_pred_tf_tuned),
        "Precision (macro)": precision_score(y_test_np, y_pred_tf_tuned, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test_np, y_pred_tf_tuned, average="macro", zero_division=0),
        "F1 (macro)": f1_score(y_test_np, y_pred_tf_tuned, average="macro", zero_division=0),
        "F1 (weighted)": f1_score(y_test_np, y_pred_tf_tuned, average="weighted", zero_division=0)
    }
])

display(results_after_optuna.round(4))


,Model,Accuracy,Precision (macro),Recall (macro),F1 (macro),F1 (weighted)
0,PyTorch + Optuna,0.5112,0.3203,0.2589,0.1865,0.3528
1,TensorFlow + Optuna,0.5235,0.2793,0.2794,0.2218,0.3788


# Result and Conclusion

In [170]:
torch_acc = accuracy_score(y_test_np, torch_test_preds)
tf_acc = accuracy_score(y_test_np, y_pred_tf_tuned)
torch_f1 = f1_score(y_test_np, torch_test_preds, average="macro", zero_division=0)
tf_f1 = f1_score(y_test_np, y_pred_tf_tuned, average="macro", zero_division=0)

print("RESULT")
print(f"PyTorch + Optuna: test accuracy={torch_acc:.4f}, macro F1={torch_f1:.4f}")
print(f"TensorFlow + Optuna: test accuracy={tf_acc:.4f}, macro F1={tf_f1:.4f}")

if tf_acc > torch_acc:
    winner = "TensorFlow"
elif torch_acc > tf_acc:
    winner = "PyTorch"
else:
    winner = "Neither; both have the same test accuracy"

print(f"\nCONCLUSION: {winner} performed better by test accuracy on this run.")
print("Use macro F1 and the per-class classification report as well, especially if the target classes are imbalanced.")
print("These results are from one run; small differences may be caused by random initialization and training variation.")


RESULT
PyTorch + Optuna: test accuracy=0.5112, macro F1=0.1865
TensorFlow + Optuna: test accuracy=0.5235, macro F1=0.2218

CONCLUSION: TensorFlow performed better by test accuracy on this run.
Use macro F1 and the per-class classification report as well, especially if the target classes are imbalanced.
These results are from one run; small differences may be caused by random initialization and training variation.
